In [35]:
import numpy as np
from lightrag import LightRAG, QueryParam
from lightrag.utils import wrap_embedding_func_with_attrs
from lightrag.llm.ollama import ollama_model_complete, ollama_embed
import os
import pandas as pd

In [36]:
cd C:\Users\angel\Desktop\Recommand-System\MOST_committee

C:\Users\angel\Desktop\Recommand-System\MOST_committee


In [37]:
WORKING_DIR = "./lightrag_storage"
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

In [38]:
OLLAMA_HOST = "http://localhost:1228"
LLM_MODEL_NAME = "gpt-oss:120b"
EMBEDDING_MODEL_NAME = "BAAI/bge-large-zh-v1.5"

In [39]:
from sentence_transformers import SentenceTransformer
import numpy as np
import asyncio
from functools import wraps
# 創建一個模型緩存
_embedding_model = None

def get_embedding_model():
    """獲取或加載嵌入模型"""
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    return _embedding_model

@wrap_embedding_func_with_attrs(embedding_dim=1024, max_token_size=8192, model_name=EMBEDDING_MODEL_NAME)
async def embedding_func(texts: list[str]) -> np.ndarray:
    """使用 HuggingFace 模型進行文本嵌入"""
    # 在非阻塞線程中運行模型推理
    def _embed():
        model = get_embedding_model()
        embeddings = model.encode(texts, convert_to_numpy=True)
        return embeddings
    
    # 使用 asyncio 在執行器中運行 CPU 密集型任務
    return await asyncio.get_event_loop().run_in_executor(None, _embed)

async def initialize_rag():
    rag = LightRAG(
        working_dir=WORKING_DIR,
        # 使用 Ollama 模型進行文本生成，並指定主機
        llm_model_func=lambda prompt, **kwargs: ollama_model_complete(
            prompt=prompt,
            model=LLM_MODEL_NAME,  # 指定模型名稱
            host=OLLAMA_HOST,  # 指定 Ollama 主機
            **kwargs
        ),
        llm_model_name=LLM_MODEL_NAME,  # 模型名稱（用於記錄）
        llm_model_kwargs={
            "options": {"num_ctx": 128000},  # 模型選項
            "format": "json",  # 如果需要 JSON 輸出
            "temperature": 0.5  # 溫度參數
        },
        embedding_func=embedding_func,  # 使用裝飾後的嵌入函數
    )
    # IMPORTANT: Both initialization calls are required!
    await rag.initialize_storages()  # Initialize storage backends
    return rag


In [40]:
pass_project_file_name = 'data/research_proj/115計算機學門審查/pass_project.xlsx'
years = ['108','109','110','111','112','113','114']

In [41]:
authors_dicts = {}  # key: author name, value: dict with titles, abstracts, keywords

for year in years:
    pass_project_df = pd.read_excel(pass_project_file_name, sheet_name=year)
    for index, row in pass_project_df.iterrows():
        author = row['計畫主持人']
        
        # 初始化作者字典及其三個子列表
        if author not in authors_dicts:
            authors_dicts[author] = []
            
        title = row['計畫中文名稱']
        abstract = row['中文摘要']
        
        # 添加項目信息
        authors_dicts[author].append(f'計畫名稱:{title}\n摘要:{abstract}\n關鍵字:{row["中文關鍵字"]}\n')
    
print(authors_dicts)


{'楊得年': ['計畫名稱:虛擬實境之社群網路群組物件推薦\n摘要:虛擬實境(Virtual Reality, VR)應用程式已逐漸融入生活，如Facebook虛擬社群、Amazon、eBay、IKEA 虛擬百貨公司等；然而資料探勘領域中VR相關研究尚稱缺乏。在實體商店中，商品配置須保持固定，難以顧及個人喜好，亦使群組推薦系統難以滿足所有使用者。現今線上購物因個人化推薦而難以吸引團體購物，亦不利於朋友互動。在VR社群購物中，場景及商品應一致以達成良好體驗；然而VR特有之多視技術 (Multi-View Display, MVD) 使少數商品可依個人喜好替換。換言之，朋友在同位置所看之商品可不相同，從而使個人更易滿足。本計畫將探討新型態VR推薦系統，其中MVD、社群關係、個人化、商品陳列等均為須考量因素。 更精確而言，本計畫將提出一社群VR商品包裹推薦系統，含二階段：1)多視商品喜好學習系統；根據使用者間之社交關係及商品切換成本，從歷史資料及消費者行為學習個人對各商品之喜好。2)多視商品包裹查詢；根據前述之MVD商品喜好，本系統考慮資料庫中個人喜好、社群關係、商品陳列以生成最佳商品包裹，亦將證明此最佳化問題之NP難度，並設計近似演算法及證明近似比例。 本計畫二階段均將蒐集真實巨量資料以進行實驗，並以 Unity 3D 實作使用者調查系統，使用HTC VIVE 頭戴式裝置評估滿意度，最後將程式碼開源並整合入現存相關開源計畫。\n關鍵字:社群網路、虛擬實境、商品推薦、社群群組查詢\n', '計畫名稱:以知識圖譜為基礎之關聯多商品社群影響力行銷\n摘要:知識圖譜(如YAGO、Freebase、DBpedia、NELL和Probase)已廣泛應用於表示物件之間的各種關係，近來亦有許多得益於知識圖譜的應用，例如相關性度量、問答搜尋和推薦系統。然而，結合知識圖譜上豐富資訊以進行社群行銷的研究卻仍十分稀少。另一方面，社群行銷與推薦系統利用社群影響力在實務上已獲得成功，在學術上也有許多研究影響力、收益和利潤最大化的問題。然而，除了社群影響力外，個人對物品的偏好與物品間的複雜關係也在使用者的購買決策中扮演舉足輕重的角色，由於使用者的需求與偏好會隨著已經購買過的物品而改變，其對物品間關係的認知也會隨之動態變化，而由於偏好與認知相似的人往往會更加親近，這些變化可能也會接著影響使用者的社群

In [42]:
rag = await initialize_rag()
for author, info in authors_dicts.items():
    document = "\n".join(info)
    await rag.ainsert(
        document,file_paths=author
    )


INFO: [] Created new empty graph file: ./lightrag_storage\graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': './lightrag_storage\\vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': './lightrag_storage\\vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': './lightrag_storage\\vdb_chunks.json'} 0 data
INFO: Reset 24 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 29 document(s)
INFO: Extracting stage 1/29: 康立威
INFO: Processing d-id: doc-0282fa5221486d31e922c74ec8b047f8
INFO: Extracting stage 2/29: 紀明德
INFO: Processing d-id: doc-65cd0fd4c06b527d1c54a85738f89956
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransforme

CancelledError: 

In [ ]:
print(dir(rag))


['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__final__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__post_init__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_get_storage_class', '_insert_done', '_migrate_chunk_tracking_storage', '_migrate_entity_relation_data', '_process_extract_entities', '_query_done', '_storages_status', '_validate_and_fix_document_consistency', 'aclear_cache', 'acreate_entity', 'acreate_relation', 'addon_params', 'adelete_by_doc_id', 'adelete_by_entity', 'adelete_by_relation', 'aedit_entity', 'aedit_relation', 'aexport_data', 'aget_docs_by_ids', 'aget_docs_by_track_id', 'ainsert', 'ainsert_custom_chunks', 'ainsert_custom_kg', 'amerge_entities', 'apipe